# ARE-002.1 - YummyAnime API Exploration
One public profile, two GET requests, public X-Application only. No Bearer, cookies,
retries, redirects, database, or production client.

Install: `python -m pip install -e ".[research]"`. Select the local .venv kernel.
Set YUMMYANIME_APPLICATION_TOKEN in the root .env. Run all cells in order.
On an API error no further requests are sent; empty tables do not mean an empty profile.

PRIVACY: profile and table outputs contain personal data during your session.
Jupyter may autosave them. Clear Outputs of All Cells before saving or sharing.
This repository copy has no saved outputs. Never export the DataFrame.

Official documentation: https://ru.yummyani.me/api/swagger
API: https://api.yani.tv. Website counters are entered manually; no site scraping.


In [ ]:
import os
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display

root = Path.cwd().resolve()
if not (root / "pyproject.toml").is_file():
    root = root.parent
if not (root / "pyproject.toml").is_file():
    raise RuntimeError("Run from the project root or notebooks directory.")
load_dotenv(root / ".env", override=False)
token = os.getenv("YUMMYANIME_APPLICATION_TOKEN", "").strip()
nickname = os.getenv("YUMMYANIME_TEST_NICKNAME", "").strip() or input("Enter your nickname: ") or "First_Light"
base_url = "https://api.yani.tv"
http_log = []
limitations = []
api_ready = bool(token)
if not api_ready:
    limitations.append("Public application token missing; no requests sent.")


def get_response(path, label):
    """No retries, redirects, cookies, body, URL, or token logging."""
    global api_ready
    if not api_ready:
        return None
    try:
        response = requests.get(
            base_url + path,
            headers={"X-Application": token, "Accept": "application/json"},
            timeout=(10, 30), allow_redirects=False,
        )
    except requests.RequestException:
        http_log.append({"endpoint": label, "HTTP": None, "result": "network error"})
        limitations.append("Network error; stopped; details omitted.")
        api_ready = False
        return None
    http_log.append({"endpoint": label, "HTTP": response.status_code,
                     "result": "OK" if response.status_code == 200 else "stopped"})
    if response.status_code != 200:
        limitations.append(f"{label}: HTTP {response.status_code}; stopped.")
        api_ready = False
        return None
    try:
        payload = response.json()
    except ValueError:
        payload = None
    if (not isinstance(payload, dict) or "response" not in payload
            or payload.get("error") or payload.get("errors")):
        limitations.append(f"{label}: invalid JSON envelope or API error; stopped.")
        api_ready = False
        return None
    return payload["response"]


## Profile
Display only ID, nickname, and exact spelling match.

In [ ]:
profile = get_response("/users/" + quote(nickname, safe=""), "GET /users/{nickname}")
user_id = profile.get("id") if isinstance(profile, dict) else None
profile_found = isinstance(user_id, int) and not isinstance(user_id, bool)
if profile_found:
    display(pd.DataFrame([{
        "user_id": user_id,
        "nickname": profile.get("nickname"),
        "exact_nickname_match": profile.get("nickname") == nickname,
    }]))
else:
    api_ready = False
    limitations.append("Profile not found or no integer user ID.")
display(pd.DataFrame(http_log))


## Lists and DataFrame
Use user.rating, never the global anime rating. Favorites may overlap watch statuses.
The documented date means list addition, not a history of status changes.


In [ ]:
rows = get_response(f"/users/{user_id}/lists", "GET /users/{id}/lists") if profile_found else None
if rows is None:
    rows = []
elif not isinstance(rows, list) or any(not isinstance(row, dict) for row in rows):
    limitations.append("Unexpected list schema; DataFrame remains empty.")
    rows = []
    api_ready = False


def nested(row, *keys):
    value = row
    for key in keys:
        if not isinstance(value, dict):
            return None
        value = value.get(key)
    return value


columns = ["anime_id", "title", "user_rating", "list_id", "status_alias", "is_fav", "date", "myanimelist_id"]
df = pd.DataFrame([{
    "anime_id": row.get("anime_id"),
    "title": row.get("title"),
    "user_rating": nested(row, "user", "rating"),
    "list_id": nested(row, "user", "list", "list", "id"),
    "status_alias": nested(row, "user", "list", "list", "href"),
    "is_fav": nested(row, "user", "list", "is_fav"),
    "date": row.get("date"),
    "myanimelist_id": nested(row, "remote_ids", "myanimelist_id"),
} for row in rows], columns=columns)
for col in ["anime_id", "list_id", "date", "myanimelist_id"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
df["user_rating"] = pd.to_numeric(df["user_rating"], errors="coerce").astype("Float64")
df["is_fav"] = df["is_fav"].map(lambda value: value if isinstance(value, bool) else pd.NA).astype("boolean")
display(pd.DataFrame(http_log))
display(df.head(10))
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum()}))
display(pd.Series({"records": len(df), "unique_anime_id": df["anime_id"].nunique(),
                   "duplicate_anime_id": int(df["anime_id"].dropna().duplicated().sum())}, name="summary"))


## Watch statuses
Keep missing and unknown statuses. Do not sum favorites with watch categories.

In [ ]:
status_names = {0: "Watching", 1: "Planned", 2: "Completed", 3: "Dropped", 5: "On hold"}
status_labels = df["list_id"].map(status_names).astype("string")
unknown = df["list_id"].notna() & status_labels.isna()
status_labels.loc[unknown] = df.loc[unknown, "list_id"].map(lambda value: f"Other ({value})")
status_labels = status_labels.fillna("No status")
counts = status_labels.value_counts()
status_table = counts.reindex(list(status_names.values()) + [label for label in counts.index if label not in status_names.values()], fill_value=0).rename_axis("Status").to_frame("API")
display(status_table)


## Ratings, favorites, MAL IDs
Positive user.rating values provisionally mean rated. Zero is reported separately
as a presumed missing rating; verify this convention and ownership against the UI.


In [ ]:
rated = df.loc[df["user_rating"].gt(0).fillna(False), "user_rating"]
mal_present = df["myanimelist_id"].gt(0).fillna(False)
rating_summary = pd.Series({
    "rated_count": len(rated),
    "rating_min": rated.min() if len(rated) else pd.NA,
    "rating_max": rated.max() if len(rated) else pd.NA,
    "zero_ratings": int(df["user_rating"].eq(0).sum()),
    "missing_ratings": int(df["user_rating"].isna().sum()),
    "favorites": int(df["is_fav"].sum()),
    "mal_id_count": int(mal_present.sum()),
    "mal_id_share": float(mal_present.mean()) if len(df) else pd.NA,
}, name="metric")
display(rating_summary)
display(rated.value_counts().sort_index().rename_axis("User rating").to_frame("Count"))


## Compare with website
Replace None counters manually and rerun this cell. A blank counter is not a match.
Before saving the repository version restore None values and clear all outputs.
Matching counters support completeness for this profile only, not a universal API guarantee.


In [ ]:
website_counts = {"Completed": None, "Planned": None, "Watching": None,
                  "Dropped": None, "On hold": None}
comparison = pd.DataFrame({
    "API": {label: int(counts.get(label, 0)) for label in website_counts},
    "Website": pd.Series(website_counts, dtype="Int64"),
})
comparison["Match"] = comparison["API"].astype("Int64").eq(comparison["Website"])
if not api_ready:
    comparison["Match"] = pd.NA
    limitations.append("API analysis incomplete: empty tables do not prove an empty profile.")
display(comparison)


## Conclusions and limits
ALS fold-in requires trained item factors, a verified MAL-to-training-item mapping,
and a policy for converting ratings, statuses, and favorites to confidence.
Planned items are not automatically positive watches. Favorites can have no watch status.
This API provides a list snapshot, not a complete event history.
Offline ALS training needs a separately licensed multi-user interaction dataset.


In [ ]:
display(pd.DataFrame(http_log))
print("API analysis completed:", bool(api_ready and profile_found))
print("Website completeness verified:", bool(api_ready and comparison["Website"].notna().all() and comparison["Match"].fillna(False).all()))
for limitation in dict.fromkeys(limitations):
    print(limitation)
print("DataFrames and responses remain in memory only. Clear all outputs before saving.")


In [ ]:
df[df["is_fav"]==True]


In [ ]:
df[df["list_id"].isnull()]